# Tutorial 02: Model Training with Config

In this tutorial, we will use the **YAML configuration** to define hyperparameters and train our model using the processed dataset. The architecture is selected via the Model Factory, allowing you to switch between models with a single line change.

## 1. Setup and Imports

In [ ]:
import os
import sys
import yaml
import pickle

# Add src to path
sys.path.append(os.path.abspath('../../src'))

from bioacoustica.training.trainer import Trainer
print("Trainer module loaded.")

## 2. Load Config and Data

In [ ]:
CONFIG_PATH = "../../configs/gibbon.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DATA_DIR = "../../data/processed"

print(f"Loading processed data from {DATA_DIR}...")
with open(os.path.join(DATA_DIR, "X.pkl"), "rb") as f:
    X = pickle.load(f)

with open(os.path.join(DATA_DIR, "Y.pkl"), "rb") as f:
    Y = pickle.load(f)

print(f"Data loaded. Input shape: {X.shape}, Samples: {len(X)}")

## 3. Select Architecture & Train Model

The **Model Factory** (`get_model`) allows to switch between architectures:

| `architecture` | Best For | Notes |
| --- | --- | --- |
| `custom_cnn` | Low resources, fast training | Grayscale 1-channel spectrograms |
| `mobilenet_v2` | Edge deployment | Auto-converts 1-ch to RGB |
| `efficientnet_b0` | Best accuracy/efficiency | Auto-converts 1-ch to RGB |

> **Change `architecture` in `configs/gibbon.yaml`** to switch models. Set `fine_tune: true` to unfreeze the pre-trained backbone.

In [ ]:
trainer = Trainer(output_dir="../../models", seed=config['training']['seed'])

arch = config['training']['architecture']
print(f"Starting training with architecture: '{arch}' for {config['training']['epochs']} epochs...")

model_path = trainer.train(
    X=X,
    Y=Y,
    class_order=config['classes']['order'],
    model_architecture=arch,
    epochs=config['training']['epochs'],
    batch_size=config['training']['batch_size'],
    dropout_rate=config['training'].get('dropout_rate', 0.3),
    fine_tune=config['training'].get('fine_tune', False)
)

print(f"\nTraining complete. Model saved at: {model_path}")